In [1]:
# Imports
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.append("/projeto")

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from conf.spark_session import get_spark_session

# Sessão do Spark com Delta Lake
spark = get_spark_session()

# Removendo os logs do notebook
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e65971a1-468f-4449-93b9-97a33a90bb9a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 11750ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |

In [2]:
# Teste de configuracao S3
spark._jsc.hadoopConfiguration().get("fs.s3a.impl")

'org.apache.hadoop.fs.s3a.S3AFileSystem'

In [3]:
# Teste de acesso ao MinIO
spark._jsc.hadoopConfiguration().get("fs.s3a.endpoint")

'http://minio:9000'

In [4]:
spark._jsc.hadoopConfiguration().get("fs.s3a.access.key")

'minio'

In [5]:
spark._jsc.hadoopConfiguration().get("fs.s3a.secret.key")

'minio123'

In [8]:
# Lendo as tabelas no datalake
spark.read.text("s3a://datalake/").show()

+-----+
|value|
+-----+
+-----+



In [11]:
# Salvando arquivos parquet de teste 
spark.range(5).write.mode("overwrite").parquet("s3a://datalake/test_parquet")

In [16]:
# Lendo parquet em um dataframe
df_parquet = spark.read.format("parquet").load("s3a://datalake/test_parquet")
df_parquet.show()

+---+
| id|
+---+
|  0|
|  1|
|  3|
|  4|
|  2|
+---+



In [13]:
# Salvando uma tabela delta de teste 
spark.range(5).write.format("delta").mode("overwrite").save("s3a://datalake/test_delta")

In [15]:
# Lendo delta em um dataframe
df_delta = spark.read.format("delta").load("s3a://datalake/test_delta")
df_delta.show()

+---+
| id|
+---+
|  1|
|  0|
|  4|
|  2|
|  3|
+---+

